In [23]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [24]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

In [25]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [26]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


##_______________________________________________________________ "" ____________________________________________________________

##_______________________________________________________________ "" ____________________________________________________________

# Treatment Inventory : Template

In [27]:
# ============================================================
# 01 — TREATMENT INVENTORY
# Historical WhatsApp Policy Audit
# ============================================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# ============================================================
# 1. LOAD DATA
# ============================================================

wa = wa
queue = queue

print("=" * 100)
print("RAW DATA")
print("=" * 100)

print(f"WhatsApp rows      : {len(wa):,}")
print(f"WhatsApp customers : {wa['customer_id'].nunique():,}")
print(f"Queue rows         : {len(queue):,}")
print(f"Queue customers    : {queue['customer_id'].nunique():,}")


# ============================================================
# 2. BASIC PREPARATION
# ============================================================

wa["sent_at"] = pd.to_datetime(wa["sent_at"])

wa = (
    wa
    .sort_values(["customer_id", "sent_at", "message_id"])
    .reset_index(drop=True)
)

wa["date"] = wa["sent_at"].dt.date
wa["month"] = wa["sent_at"].dt.to_period("M").astype(str)
wa["day"] = wa["sent_at"].dt.day
wa["hour"] = wa["sent_at"].dt.hour
wa["minute"] = wa["sent_at"].dt.minute
wa["day_of_week"] = wa["sent_at"].dt.day_name()
wa["is_weekend"] = wa["sent_at"].dt.dayofweek >= 5


# chronological message number
wa["customer_message_number"] = (
    wa.groupby("customer_id").cumcount() + 1
)


# ============================================================
# 3. DATA QUALITY
# ============================================================

print("\n" + "=" * 100)
print("DATA QUALITY")
print("=" * 100)

quality = pd.DataFrame({
    "dtype": wa.dtypes.astype(str),
    "missing": wa.isna().sum(),
    "missing_pct": wa.isna().mean() * 100,
    "n_unique": wa.nunique(dropna=False)
})

print(quality)


print("\nDate range:")
print("Min:", wa["sent_at"].min())
print("Max:", wa["sent_at"].max())


# ============================================================
# 4. TEMPLATE INVENTORY
# ============================================================

print("\n" + "=" * 100)
print("TEMPLATE INVENTORY")
print("=" * 100)

template_inventory = (
    wa.groupby("template", dropna=False)
    .agg(
        messages=("message_id", "count"),
        customers=("customer_id", "nunique"),
        avg_dpd=("days_past_due", "mean"),
        median_dpd=("days_past_due", "median"),
        avg_balance=("outstanding_balance_brl", "mean"),
        avg_pressure_14d=("n_msgs_last_14d", "mean")
    )
    .reset_index()
)

template_inventory["share_messages_pct"] = (
    100 * template_inventory["messages"] / len(wa)
)

template_inventory = template_inventory.sort_values(
    "messages",
    ascending=False
)

print(template_inventory.to_string(index=False))


# ============================================================
# 5. TEMPLATE × MONTH
# ============================================================

print("\n" + "=" * 100)
print("TEMPLATE × MONTH — COUNTS")
print("=" * 100)

template_month = pd.crosstab(
    wa["template"],
    wa["month"],
    margins=True
)

print(template_month)


print("\n" + "=" * 100)
print("TEMPLATE × MONTH — COLUMN %")
print("=" * 100)

template_month_pct = (
    pd.crosstab(
        wa["template"],
        wa["month"],
        normalize="columns"
    ) * 100
)

print(template_month_pct.round(2))


# ============================================================
# 6. MESSAGE NUMBER INVENTORY
# ============================================================

print("\n" + "=" * 100)
print("MESSAGE NUMBER WITHIN CUSTOMER JOURNEY")
print("=" * 100)

message_number = (
    wa.groupby("customer_message_number")
    .agg(
        messages=("message_id", "count"),
        customers=("customer_id", "nunique")
    )
    .reset_index()
)

message_number["share_messages_pct"] = (
    message_number["messages"] / len(wa) * 100
)

print(message_number.to_string(index=False))


# ============================================================
# 7. TEMPLATE × MESSAGE NUMBER
# ============================================================

print("\n" + "=" * 100)
print("TEMPLATE × MESSAGE NUMBER — ROW %")
print("=" * 100)

template_sequence = pd.crosstab(
    wa["customer_message_number"],
    wa["template"],
    normalize="index"
) * 100

print(template_sequence.round(1).head(20))


# ============================================================
# 8. DPD DISTRIBUTION BY TEMPLATE
# ============================================================

print("\n" + "=" * 100)
print("DPD DISTRIBUTION BY TEMPLATE")
print("=" * 100)

dpd_template = (
    wa.groupby("template")["days_past_due"]
    .describe(
        percentiles=[.05, .10, .25, .50, .75, .90, .95]
    )
)

print(dpd_template)


# ============================================================
# 9. CREATE DPD BUCKET
# ============================================================

dpd_bins = [
    -np.inf,
    7,
    15,
    30,
    45,
    60,
    np.inf
]

dpd_labels = [
    "01-07",
    "08-15",
    "16-30",
    "31-45",
    "46-60",
    "60+"
]

wa["dpd_bucket"] = pd.cut(
    wa["days_past_due"],
    bins=dpd_bins,
    labels=dpd_labels
)


print("\n" + "=" * 100)
print("TEMPLATE × DPD BUCKET — COUNTS")
print("=" * 100)

template_dpd_count = pd.crosstab(
    wa["dpd_bucket"],
    wa["template"]
)

print(template_dpd_count)


print("\n" + "=" * 100)
print("TEMPLATE × DPD BUCKET — ROW %")
print("=" * 100)

template_dpd_pct = (
    pd.crosstab(
        wa["dpd_bucket"],
        wa["template"],
        normalize="index"
    ) * 100
)

print(template_dpd_pct.round(2))


# ============================================================
# 10. PRESSURE DISTRIBUTION BY TEMPLATE
# ============================================================

print("\n" + "=" * 100)
print("CONTACT PRESSURE BY TEMPLATE")
print("=" * 100)

pressure_template = (
    wa.groupby("template")["n_msgs_last_14d"]
    .describe(
        percentiles=[.10, .25, .50, .75, .90, .95]
    )
)

print(pressure_template)


# ============================================================
# 11. DAY OF WEEK × TEMPLATE
# ============================================================

dow_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

print("\n" + "=" * 100)
print("DAY OF WEEK × TEMPLATE — ROW %")
print("=" * 100)

dow_template = (
    pd.crosstab(
        wa["day_of_week"],
        wa["template"],
        normalize="index"
    )
    .reindex(dow_order)
    * 100
)

print(dow_template.round(2))


# ============================================================
# 12. HOUR × TEMPLATE
# ============================================================

print("\n" + "=" * 100)
print("HOUR × TEMPLATE — COUNTS")
print("=" * 100)

hour_template = pd.crosstab(
    wa["hour"],
    wa["template"]
)

print(hour_template)


print("\n" + "=" * 100)
print("HOUR × TEMPLATE — ROW %")
print("=" * 100)

hour_template_pct = (
    pd.crosstab(
        wa["hour"],
        wa["template"],
        normalize="index"
    ) * 100
)

print(hour_template_pct.round(2))


# ============================================================
# 13. FIRST MESSAGE TEMPLATE
# ============================================================

first_message = (
    wa
    .loc[wa["customer_message_number"] == 1]
    .copy()
)

print("\n" + "=" * 100)
print("FIRST MESSAGE TEMPLATE")
print("=" * 100)

first_template = (
    first_message["template"]
    .value_counts(dropna=False)
    .to_frame("customers")
)

first_template["pct"] = (
    first_template["customers"]
    / len(first_message)
    * 100
)

print(first_template)


# ============================================================
# 14. NUMBER OF DIFFERENT TEMPLATES PER CUSTOMER
# ============================================================

customer_treatment_diversity = (
    wa.groupby("customer_id")
    .agg(
        messages=("message_id", "count"),
        different_templates=("template", "nunique"),
        min_dpd=("days_past_due", "min"),
        max_dpd=("days_past_due", "max")
    )
)

print("\n" + "=" * 100)
print("TREATMENT DIVERSITY PER CUSTOMER")
print("=" * 100)

print(
    customer_treatment_diversity[
        "different_templates"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 15. UNIQUE TEMPLATE SEQUENCES
# ============================================================

customer_sequences = (
    wa.groupby("customer_id")["template"]
    .apply(lambda x: " -> ".join(x.astype(str)))
    .reset_index(name="template_sequence")
)

sequence_summary = (
    customer_sequences["template_sequence"]
    .value_counts()
    .head(30)
    .reset_index()
)

sequence_summary.columns = [
    "template_sequence",
    "customers"
]

sequence_summary["pct_customers"] = (
    sequence_summary["customers"]
    / wa["customer_id"].nunique()
    * 100
)

print("\n" + "=" * 100)
print("TOP 30 HISTORICAL TEMPLATE SEQUENCES")
print("=" * 100)

print(sequence_summary.to_string(index=False))


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("TREATMENT INVENTORY SUMMARY")
print("=" * 100)

print(
    f"""
Messages                     : {len(wa):,}
Customers                    : {wa['customer_id'].nunique():,}
Templates                    : {wa['template'].nunique():,}
Months                       : {wa['month'].nunique():,}
Observed hours               : {wa['hour'].nunique():,}
Observed weekdays            : {wa['day_of_week'].nunique():,}

Median messages/customer     : {wa.groupby('customer_id').size().median():.1f}
Mean messages/customer       : {wa.groupby('customer_id').size().mean():.2f}
Max messages/customer        : {wa.groupby('customer_id').size().max():,}

Median templates/customer    : {customer_treatment_diversity['different_templates'].median():.1f}
Mean templates/customer      : {customer_treatment_diversity['different_templates'].mean():.2f}
Max templates/customer       : {customer_treatment_diversity['different_templates'].max():,}
"""
)

RAW DATA
WhatsApp rows      : 75,406
WhatsApp customers : 11,724
Queue rows         : 10,658
Queue customers    : 10,658

DATA QUALITY
                                    dtype  missing  missing_pct  n_unique
message_id                         object        0         0.00     75406
customer_id                        object        0         0.00     11724
sent_at                    datetime64[ns]        0         0.00     39466
template                           object        0         0.00         4
n_msgs_last_14d                     int64        0         0.00        11
days_past_due                       int64        0         0.00        60
outstanding_balance_brl           float64        0         0.00     11510
monthly_salary_brl                float64        0         0.00       628
payday_day_of_month                 int64        0         0.00         7
n_prior_transactions                int64        0         0.00        40
account_age_months                  int64        0 

| DPD   |  Friendly |   Pix |    Urgent |  Discount |
| ----- | --------: | ----: | --------: | --------: |
| 1–7   | **69,9%** | 30,1% |    **0%** |    **0%** |
| 8–15  |     19,8% | 30,1% | **50,1%** |    **0%** |
| 16–30 |     20,1% | 29,9% | **50,1%** |    **0%** |
| 31–45 |    **0%** | 20,2% |     29,6% | **50,2%** |
| 46–60 |    **0%** | 20,5% |     29,8% | **49,7%** |

#### Com esse histórico, não permite observacionalmente : Não existe suporte ou evidências em alguns casos, como : discount <=30
#### treatment learning vai estar dentro das regiões de overlap


Pix parece quase funcionar como uma espécie de ação transversal : quase flat

early collections : Friendly / pix (70/30)  
mid collections : Friendly / pix / urgent (20/30/50)  
late collections : Pix / urgent / discount (20/30/50)  

## Quanto da ação histórica conseguimos prever usando apenas informações disponíveis antes da mensagem?

o template era escolhido a partir das características do cliente?

classificação multiclasse baseado em gradient boosting.

In [7]:
# ============================================================
# 02 — HISTORICAL POLICY AUDIT
# Can we predict which treatment the historical policy chose?
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    confusion_matrix,
    classification_report
)
from sklearn.inspection import permutation_importance


# ============================================================
# 1. COPY BASE
# ============================================================

policy = wa.copy()


# ============================================================
# 2. CREATE ONLY PRE-TREATMENT FEATURES
# ============================================================

# IMPORTANT:
# No delivery_status
# No interaction
# No paid_within_72h
# No amount_paid_brl
#
# Those happen AFTER treatment and would create leakage.


# Calendar variables known at decision time
policy["dow"] = policy["sent_at"].dt.dayofweek
policy["day_of_month"] = policy["sent_at"].dt.day
policy["hour"] = policy["sent_at"].dt.hour


# ------------------------------------------------------------
# Payday distance
# Simple calendar approximation.
# ------------------------------------------------------------

policy["days_to_payday_raw"] = (
    policy["payday_day_of_month"]
    - policy["day_of_month"]
)

# circular monthly approximation
policy["days_to_payday"] = np.where(
    policy["days_to_payday_raw"] >= 0,
    policy["days_to_payday_raw"],
    policy["days_to_payday_raw"] + 30
)


# affordability
policy["balance_to_salary"] = (
    policy["outstanding_balance_brl"]
    / policy["monthly_salary_brl"].replace(0, np.nan)
)


# ============================================================
# 3. FEATURE SET A — STATE ONLY
# ============================================================

features_state = [
    "days_past_due",
    "outstanding_balance_brl",
    "monthly_salary_brl",
    "balance_to_salary",
    "payday_day_of_month",
    "days_to_payday",
    "n_prior_transactions",
    "account_age_months",
    "days_since_last_app_login",
    "state_uf",
    "dow",
    "hour"
]


# ============================================================
# 4. FEATURE SET B — STATE + JOURNEY HISTORY
# ============================================================

features_history = features_state + [
    "n_msgs_last_14d",
    "customer_message_number"
]


TARGET = "template"


# ============================================================
# 5. TEMPORAL SPLIT
# ============================================================
#
# Train = June + July
# Test  = August
#
# This is much better than random split for our question.
# ============================================================

train = policy[
    policy["sent_at"] < "2026-08-01"
].copy()

test = policy[
    policy["sent_at"] >= "2026-08-01"
].copy()


print("=" * 100)
print("TEMPORAL SPLIT")
print("=" * 100)

print(f"Train rows : {len(train):,}")
print(f"Test rows  : {len(test):,}")

print("\nTrain treatments:")
print(train[TARGET].value_counts(normalize=True).round(4))

print("\nTest treatments:")
print(test[TARGET].value_counts(normalize=True).round(4))


# ============================================================
# 6. MODEL FUNCTION
# ============================================================

def run_policy_model(features, model_name):

    X_train = train[features].copy()
    y_train = train[TARGET].copy()

    X_test = test[features].copy()
    y_test = test[TARGET].copy()

    categorical = [
        c for c in features
        if X_train[c].dtype == "object"
    ]

    numeric = [
        c for c in features
        if c not in categorical
    ]

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipe,
            numeric
        ),
        (
            "cat",
            categorical_pipe,
            categorical
        )
    ])

    model = HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=200,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )

    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)

    classes = pipe.named_steps["model"].classes_

    print("\n" + "=" * 100)
    print(model_name)
    print("=" * 100)

    print(
        f"Accuracy          : "
        f"{accuracy_score(y_test, pred):.4f}"
    )

    print(
        f"Balanced accuracy : "
        f"{balanced_accuracy_score(y_test, pred):.4f}"
    )

    print(
        f"Macro F1          : "
        f"{f1_score(y_test, pred, average='macro'):.4f}"
    )

    print(
        f"Log loss          : "
        f"{log_loss(y_test, proba, labels=classes):.4f}"
    )

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            pred,
            digits=3
        )
    )

    cm = pd.DataFrame(
        confusion_matrix(
            y_test,
            pred,
            labels=classes
        ),
        index=[
            f"actual_{x}"
            for x in classes
        ],
        columns=[
            f"pred_{x}"
            for x in classes
        ]
    )

    print("\nConfusion matrix:")
    print(cm)

    # --------------------------------------------------------
    # Propensity predictions
    # --------------------------------------------------------

    propensity = pd.DataFrame(
        proba,
        columns=[
            f"p_{c}"
            for c in classes
        ],
        index=test.index
    )

    propensity["actual_treatment"] = y_test

    # probability assigned to treatment actually observed
    class_to_idx = {
        c: i
        for i, c in enumerate(classes)
    }

    actual_idx = np.array([
        class_to_idx[x]
        for x in y_test
    ])

    propensity["p_observed_treatment"] = (
        proba[
            np.arange(len(proba)),
            actual_idx
        ]
    )

    print("\nObserved-treatment propensity:")
    print(
        propensity[
            "p_observed_treatment"
        ].describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )

    return pipe, propensity


# ============================================================
# 7. MODEL A — STATE ONLY
# ============================================================

model_state, propensity_state = run_policy_model(
    features_state,
    "MODEL A — STATE ONLY"
)


# ============================================================
# 8. MODEL B — STATE + HISTORY
# ============================================================

model_history, propensity_history = run_policy_model(
    features_history,
    "MODEL B — STATE + JOURNEY HISTORY"
)


# ============================================================
# 9. PROPENSITY DISTRIBUTION BY ACTUAL TREATMENT
# ============================================================

print("\n" + "=" * 100)
print("PROPENSITY OF OBSERVED TREATMENT — BY ACTUAL TEMPLATE")
print("=" * 100)

propensity_summary = (
    propensity_history
    .groupby("actual_treatment")[
        "p_observed_treatment"
    ]
    .describe(
        percentiles=[
            .05,
            .10,
            .25,
            .50,
            .75,
            .90,
            .95
        ]
    )
)

print(propensity_summary)


# ============================================================
# 10. HOW OFTEN IS POLICY NEAR-DETERMINISTIC?
# ============================================================

prob_cols = [
    c for c in propensity_history.columns
    if c.startswith("p_")
    and c != "p_observed_treatment"
]

propensity_history["max_action_probability"] = (
    propensity_history[
        prob_cols
    ].max(axis=1)
)


print("\n" + "=" * 100)
print("POLICY DETERMINISM")
print("=" * 100)

for threshold in [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.95
]:

    pct = (
        propensity_history[
            "max_action_probability"
        ]
        .ge(threshold)
        .mean()
        * 100
    )

    print(
        f"P(max treatment probability >= "
        f"{threshold:.2f}) = {pct:.2f}%"
    )

TEMPORAL SPLIT
Train rows : 44,181
Test rows  : 31,225

Train treatments:
template
friendly_reminder   0.36
urgent_reminder     0.30
pix_link            0.29
discount_offer      0.05
Name: proportion, dtype: float64

Test treatments:
template
urgent_reminder     0.32
pix_link            0.28
friendly_reminder   0.27
discount_offer      0.13
Name: proportion, dtype: float64

MODEL A — STATE ONLY
Accuracy          : 0.5392
Balanced accuracy : 0.5768
Macro F1          : 0.4794
Log loss          : 0.9402

Classification report:
                   precision    recall  f1-score   support

   discount_offer      0.498     0.876     0.635      3992
friendly_reminder      0.691     0.639     0.664      8543
         pix_link      0.319     0.010     0.018      8725
  urgent_reminder      0.486     0.782     0.599      9965

         accuracy                          0.539     31225
        macro avg      0.499     0.577     0.479     31225
     weighted avg      0.497     0.539     0.459     31

Classification report:
                   precision    recall  f1-score   support

   discount_offer      0.500     0.880     0.638      3992
friendly_reminder      0.691     0.640     0.665      8543
         pix_link      0.260     0.007     0.014      8725
  urgent_reminder      0.485     0.782     0.599      9965

* Discount offer : -> recall 88 é altamente previsivel a partir de X : DPD > 30 de todos as mensagens que sao discount, quantas o modelo previu discount
* friendly_reminder : -> recall 64 é razoavelmente previsivel a partir de X : DPD 1-7ta va
* pix link : -> recall 0,7 o modelo nunca diz proque ele é cross

## Dentro de cada região de DPD, clientes comparáveis tinham chance real de receber cada um dos treatments disponíveis?

DPD 1−7 : Friendly vs Pix  
DPD 8−30 : Friendly vs Pix vs Urgent  
DPD 31−60 : Pix vs Urgent vs Discount    


In [10]:
# ============================================================
# 03 — COMMON SUPPORT / POSITIVITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss


# ============================================================
# 1. PREPARE BASE
# ============================================================

support = wa.copy()

support["month"] = support["sent_at"].dt.to_period("M").astype(str)
support["dow"] = support["sent_at"].dt.dayofweek
support["day_of_month"] = support["sent_at"].dt.day

# affordability
support["balance_to_salary"] = (
    support["outstanding_balance_brl"]
    / support["monthly_salary_brl"].replace(0, np.nan)
)

# approximate distance to next payday
support["days_to_payday_raw"] = (
    support["payday_day_of_month"]
    - support["day_of_month"]
)

support["days_to_payday"] = np.where(
    support["days_to_payday_raw"] >= 0,
    support["days_to_payday_raw"],
    support["days_to_payday_raw"] + 30
)


# ============================================================
# 2. FEATURES
# ============================================================
#
# Only information available BEFORE treatment.
#
# Notice:
# - no delivery_status
# - no interaction
# - no paid_within_72h
# - no amount_paid_brl
# - no hour for now
#
# Month is included because we found policy drift.
# ============================================================

FEATURES = [
    "days_past_due",
    "outstanding_balance_brl",
    "monthly_salary_brl",
    "balance_to_salary",
    "payday_day_of_month",
    "days_to_payday",
    "n_prior_transactions",
    "account_age_months",
    "days_since_last_app_login",
    "state_uf",
    "dow",
    "month",
    "n_msgs_last_14d",
    "customer_message_number"
]


# ============================================================
# 3. DEFINE THE THREE SUPPORT REGIONS
# ============================================================

regions = {

    "EARLY_DPD_01_07": {
        "min_dpd": 1,
        "max_dpd": 7,
        "treatments": [
            "friendly_reminder",
            "pix_link"
        ]
    },

    "MID_DPD_08_30": {
        "min_dpd": 8,
        "max_dpd": 30,
        "treatments": [
            "friendly_reminder",
            "pix_link",
            "urgent_reminder"
        ]
    },

    "LATE_DPD_31_60": {
        "min_dpd": 31,
        "max_dpd": 60,
        "treatments": [
            "pix_link",
            "urgent_reminder",
            "discount_offer"
        ]
    }
}


# ============================================================
# 4. PROPENSITY MODEL
# ============================================================

def fit_region_propensity(df, region_name, config):

    region = df[
        df["days_past_due"].between(
            config["min_dpd"],
            config["max_dpd"]
        )
        &
        df["template"].isin(
            config["treatments"]
        )
    ].copy()

    print("\n")
    print("=" * 110)
    print(region_name)
    print("=" * 110)

    print(
        f"DPD range  : "
        f"{config['min_dpd']}–{config['max_dpd']}"
    )

    print(
        f"Rows       : {len(region):,}"
    )

    print(
        f"Customers  : "
        f"{region['customer_id'].nunique():,}"
    )

    print("\nTreatment distribution:")

    dist = pd.DataFrame({
        "messages": region["template"].value_counts(),
        "share": region["template"].value_counts(normalize=True)
    })

    print(dist)


    # --------------------------------------------------------
    # Train on Jun-Jul
    # Score August
    # --------------------------------------------------------

    train_r = region[
        region["sent_at"] < "2026-08-01"
    ].copy()

    test_r = region[
        region["sent_at"] >= "2026-08-01"
    ].copy()

    print("\nTemporal split:")
    print(f"Train : {len(train_r):,}")
    print(f"Test  : {len(test_r):,}")


    X_train = train_r[FEATURES]
    y_train = train_r["template"]

    X_test = test_r[FEATURES]
    y_test = test_r["template"]


    categorical = [
        c for c in FEATURES
        if X_train[c].dtype == "object"
    ]

    numeric = [
        c for c in FEATURES
        if c not in categorical
    ]


    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    prep = ColumnTransformer([
        (
            "num",
            numeric_pipe,
            numeric
        ),
        (
            "cat",
            categorical_pipe,
            categorical
        )
    ])


    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=250,
        max_leaf_nodes=15,
        min_samples_leaf=50,
        l2_regularization=2.0,
        random_state=42
    )


    pipe = Pipeline([
        ("prep", prep),
        ("model", model)
    ])


    pipe.fit(
        X_train,
        y_train
    )


    # --------------------------------------------------------
    # Predict propensity
    # --------------------------------------------------------

    proba = pipe.predict_proba(X_test)

    classes = pipe.named_steps[
        "model"
    ].classes_


    prop = pd.DataFrame(
        proba,
        columns=[
            f"p_{c}"
            for c in classes
        ],
        index=test_r.index
    )


    prop["actual_treatment"] = y_test
    prop["customer_id"] = test_r["customer_id"]
    prop["days_past_due"] = test_r["days_past_due"]


    # probability of treatment actually received
    class_map = {
        c: i
        for i, c in enumerate(classes)
    }

    actual_idx = np.array([
        class_map[x]
        for x in y_test
    ])

    prop["p_observed"] = proba[
        np.arange(len(proba)),
        actual_idx
    ]


    # --------------------------------------------------------
    # Overall propensity distributions
    # --------------------------------------------------------

    print("\n" + "-" * 110)
    print("PROPENSITY DISTRIBUTION")
    print("-" * 110)

    probability_cols = [
        f"p_{c}"
        for c in classes
    ]

    print(
        prop[
            probability_cols
        ].describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        ).T
    )


    # --------------------------------------------------------
    # Propensity of actual treatment
    # --------------------------------------------------------

    print("\n" + "-" * 110)
    print("PROPENSITY OF ACTUAL TREATMENT")
    print("-" * 110)

    print(
        prop.groupby(
            "actual_treatment"
        )["p_observed"]
        .describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )


    # --------------------------------------------------------
    # Positivity thresholds
    # --------------------------------------------------------

    print("\n" + "-" * 110)
    print("POSITIVITY / COMMON SUPPORT")
    print("-" * 110)

    thresholds = [
        0.01,
        0.025,
        0.05,
        0.10
    ]

    positivity_rows = []

    for treatment in classes:

        col = f"p_{treatment}"

        for threshold in thresholds:

            share_above = (
                prop[col] >= threshold
            ).mean()

            positivity_rows.append({
                "treatment": treatment,
                "threshold": threshold,
                "pct_population_with_propensity_above_threshold":
                    100 * share_above
            })


    positivity_table = pd.DataFrame(
        positivity_rows
    )

    print(
        positivity_table.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Strict common support
    #
    # Every available treatment must have propensity
    # >= threshold.
    # --------------------------------------------------------

    print("\n" + "-" * 110)
    print("STRICT COMMON SUPPORT")
    print("-" * 110)

    strict_rows = []

    for threshold in thresholds:

        in_support = (
            prop[
                probability_cols
            ].min(axis=1)
            >= threshold
        )

        strict_rows.append({
            "threshold": threshold,
            "rows": int(in_support.sum()),
            "pct_rows": 100 * in_support.mean(),
            "customers": prop.loc[
                in_support,
                "customer_id"
            ].nunique()
        })


    strict_support = pd.DataFrame(
        strict_rows
    )

    print(
        strict_support.to_string(
            index=False
        )
    )


    # --------------------------------------------------------
    # Extreme propensities
    # --------------------------------------------------------

    prop["min_propensity"] = (
        prop[
            probability_cols
        ].min(axis=1)
    )

    prop["max_propensity"] = (
        prop[
            probability_cols
        ].max(axis=1)
    )


    print("\n" + "-" * 110)
    print("MINIMUM PROPENSITY ACROSS AVAILABLE ACTIONS")
    print("-" * 110)

    print(
        prop[
            "min_propensity"
        ].describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )


    print("\n" + "-" * 110)
    print("MAXIMUM PROPENSITY ACROSS AVAILABLE ACTIONS")
    print("-" * 110)

    print(
        prop[
            "max_propensity"
        ].describe(
            percentiles=[
                .01,
                .05,
                .10,
                .25,
                .50,
                .75,
                .90,
                .95,
                .99
            ]
        )
    )


    return {
        "region": region,
        "train": train_r,
        "test": test_r,
        "model": pipe,
        "propensity": prop,
        "strict_support": strict_support,
        "positivity": positivity_table
    }


# ============================================================
# 5. RUN ALL REGIONS
# ============================================================

support_results = {}

for region_name, config in regions.items():

    support_results[
        region_name
    ] = fit_region_propensity(
        support,
        region_name,
        config
    )



EARLY_DPD_01_07
DPD range  : 1–7
Rows       : 23,724
Customers  : 11,019

Treatment distribution:
                   messages  share
template                          
friendly_reminder     16583   0.70
pix_link               7141   0.30

Temporal split:
Train : 15,826
Test  : 7,898

--------------------------------------------------------------------------------------------------------------
PROPENSITY DISTRIBUTION
--------------------------------------------------------------------------------------------------------------
                       count  mean  std  min   1%   5%  10%  25%  50%  75%  90%  95%  99%  max
p_friendly_reminder 7,898.00  0.70 0.01 0.63 0.66 0.68 0.69 0.70 0.70 0.71 0.72 0.72 0.73 0.75
p_pix_link          7,898.00  0.30 0.01 0.25 0.27 0.28 0.28 0.29 0.30 0.30 0.31 0.32 0.34 0.37

--------------------------------------------------------------------------------------------------------------
PROPENSITY OF ACTUAL TREATMENT
---------------------------------------

## O histórico contém tanto clientes semelhantes que receberam um tratamento quanto clientes semelhantes que receberam outro tratamento
então é possivel comparar para tentar inferir qual tratamemnto é melhor

Sugere que dentro da região de DPD, o template era distribuído com probabilidades relativamente estáveis.

## Antes da mensagem os clientes dos diferentes tratamentos eram parecidos ?

cuidado : 75 mil observações diferenças pequenas podem ficar estatisticamente significativas
standardized é melhor pra evitar.

In [14]:
# ============================================================
# 04 — COVARIATE BALANCE AUDIT
# BEFORE PROPENSITY WEIGHTING
# ============================================================

import pandas as pd
import numpy as np
from itertools import combinations


# ============================================================
# 1. BASE
# ============================================================

balance = support.copy()


# ============================================================
# 2. NUMERIC PRE-TREATMENT COVARIATES
# ============================================================

NUMERIC_FEATURES = [
    "days_past_due",
    "outstanding_balance_brl",
    "monthly_salary_brl",
    "balance_to_salary",
    "payday_day_of_month",
    "days_to_payday",
    "n_prior_transactions",
    "account_age_months",
    "days_since_last_app_login",
    "n_msgs_last_14d",
    "customer_message_number"
]


# ============================================================
# 3. SMD FUNCTION
# ============================================================

def standardized_mean_difference(x1, x2):

    x1 = pd.to_numeric(x1, errors="coerce").dropna()
    x2 = pd.to_numeric(x2, errors="coerce").dropna()

    mean1 = x1.mean()
    mean2 = x2.mean()

    var1 = x1.var(ddof=1)
    var2 = x2.var(ddof=1)

    pooled_sd = np.sqrt(
        (var1 + var2) / 2
    )

    if pooled_sd == 0 or np.isnan(pooled_sd):
        return np.nan

    return (
        (mean1 - mean2)
        / pooled_sd
    )


# ============================================================
# 4. NUMERIC BALANCE TABLE
# ============================================================

def numeric_balance_table(
    df,
    treatments,
    region_name
):

    rows = []

    treatment_pairs = list(
        combinations(
            treatments,
            2
        )
    )

    for t1, t2 in treatment_pairs:

        d1 = df[
            df["template"] == t1
        ]

        d2 = df[
            df["template"] == t2
        ]

        for feature in NUMERIC_FEATURES:

            smd = standardized_mean_difference(
                d1[feature],
                d2[feature]
            )

            rows.append({

                "region": region_name,

                "treatment_1": t1,
                "treatment_2": t2,

                "feature": feature,

                "mean_t1": d1[feature].mean(),
                "mean_t2": d2[feature].mean(),

                "median_t1": d1[feature].median(),
                "median_t2": d2[feature].median(),

                "smd": smd,
                "abs_smd": abs(smd)
                    if pd.notna(smd)
                    else np.nan
            })


    result = pd.DataFrame(rows)

    result["balance_flag"] = pd.cut(
        result["abs_smd"],
        bins=[
            -np.inf,
            0.10,
            0.20,
            0.50,
            np.inf
        ],
        labels=[
            "GOOD <0.10",
            "WATCH 0.10-0.20",
            "IMBALANCED 0.20-0.50",
            "SEVERE >0.50"
        ]
    )

    return result


# ============================================================
# 5. CATEGORICAL SMD
#
# For categorical variables:
# each category becomes a binary indicator.
#
# Example:
# state_SP = 1/0
# month_2026-08 = 1/0
# ============================================================

CATEGORICAL_FEATURES = [
    "state_uf",
    "dow",
    "month"
]


def binary_smd(p1, p2):

    pooled = (
        p1 * (1 - p1)
        +
        p2 * (1 - p2)
    ) / 2

    pooled_sd = np.sqrt(pooled)

    if pooled_sd == 0:
        return np.nan

    return (
        (p1 - p2)
        / pooled_sd
    )


def categorical_balance_table(
    df,
    treatments,
    region_name
):

    rows = []

    treatment_pairs = list(
        combinations(
            treatments,
            2
        )
    )

    for feature in CATEGORICAL_FEATURES:

        categories = sorted(
            df[feature]
            .dropna()
            .unique()
        )

        for category in categories:

            for t1, t2 in treatment_pairs:

                d1 = df[
                    df["template"] == t1
                ]

                d2 = df[
                    df["template"] == t2
                ]

                p1 = (
                    d1[feature] == category
                ).mean()

                p2 = (
                    d2[feature] == category
                ).mean()

                smd = binary_smd(
                    p1,
                    p2
                )

                rows.append({

                    "region": region_name,

                    "treatment_1": t1,
                    "treatment_2": t2,

                    "feature":
                        f"{feature}={category}",

                    "mean_t1": p1,
                    "mean_t2": p2,

                    "median_t1": np.nan,
                    "median_t2": np.nan,

                    "smd": smd,

                    "abs_smd":
                        abs(smd)
                        if pd.notna(smd)
                        else np.nan
                })


    result = pd.DataFrame(rows)

    result["balance_flag"] = pd.cut(
        result["abs_smd"],
        bins=[
            -np.inf,
            0.10,
            0.20,
            0.50,
            np.inf
        ],
        labels=[
            "GOOD <0.10",
            "WATCH 0.10-0.20",
            "IMBALANCED 0.20-0.50",
            "SEVERE >0.50"
        ]
    )

    return result


# ============================================================
# 6. RUN EACH REGION
# ============================================================

balance_results = {}


for region_name, config in regions.items():

    region_df = balance[
        balance["days_past_due"].between(
            config["min_dpd"],
            config["max_dpd"]
        )
        &
        balance["template"].isin(
            config["treatments"]
        )
    ].copy()


    numeric_result = (
        numeric_balance_table(
            region_df,
            config["treatments"],
            region_name
        )
    )


    categorical_result = (
        categorical_balance_table(
            region_df,
            config["treatments"],
            region_name
        )
    )


    full_result = pd.concat(
        [
            numeric_result,
            categorical_result
        ],
        ignore_index=True
    )


    balance_results[
        region_name
    ] = full_result


# ============================================================
# 7. PRINT DETAILED NUMERIC RESULTS
# ============================================================

for region_name in regions.keys():

    result = balance_results[
        region_name
    ]

    numeric_only = result[
        result["feature"].isin(
            NUMERIC_FEATURES
        )
    ].copy()

    print("\n")
    print("=" * 120)
    print(region_name)
    print("NUMERIC COVARIATE BALANCE")
    print("=" * 120)

    print(
        numeric_only[
            [
                "treatment_1",
                "treatment_2",
                "feature",
                "mean_t1",
                "mean_t2",
                "smd",
                "abs_smd",
                "balance_flag"
            ]
        ]
        .sort_values(
            "abs_smd",
            ascending=False
        )
        .to_string(
            index=False
        )
    )


# ============================================================
# 8. WORST IMBALANCES — ALL FEATURES
# ============================================================

for region_name in regions.keys():

    result = balance_results[
        region_name
    ]

    print("\n")
    print("=" * 120)
    print(region_name)
    print("TOP 25 WORST COVARIATE IMBALANCES")
    print("=" * 120)

    print(
        result[
            [
                "treatment_1",
                "treatment_2",
                "feature",
                "mean_t1",
                "mean_t2",
                "abs_smd",
                "balance_flag"
            ]
        ]
        .sort_values(
            "abs_smd",
            ascending=False
        )
        .head(25)
        .to_string(
            index=False
        )
    )


# ============================================================
# 9. SUMMARY BY REGION
# ============================================================

summary_rows = []

for region_name in regions.keys():

    result = balance_results[
        region_name
    ].dropna(
        subset=["abs_smd"]
    )

    summary_rows.append({

        "region":
            region_name,

        "comparisons":
            len(result),

        "mean_abs_smd":
            result["abs_smd"].mean(),

        "median_abs_smd":
            result["abs_smd"].median(),

        "max_abs_smd":
            result["abs_smd"].max(),

        "pct_good_lt_010":
            100 * (
                result["abs_smd"] < 0.10
            ).mean(),

        "pct_watch_010_020":
            100 * (
                (
                    result["abs_smd"] >= 0.10
                )
                &
                (
                    result["abs_smd"] < 0.20
                )
            ).mean(),

        "pct_imbalanced_ge_020":
            100 * (
                result["abs_smd"] >= 0.20
            ).mean(),

        "pct_severe_ge_050":
            100 * (
                result["abs_smd"] >= 0.50
            ).mean()
    })


balance_summary = pd.DataFrame(
    summary_rows
)


print("\n")
print("=" * 120)
print("OVERALL COVARIATE BALANCE SUMMARY")
print("=" * 120)

print(
    balance_summary
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 10. MAX SMD BY TREATMENT PAIR
# ============================================================

all_balance = pd.concat(
    balance_results.values(),
    ignore_index=True
)


pair_summary = (
    all_balance
    .dropna(
        subset=["abs_smd"]
    )
    .groupby(
        [
            "region",
            "treatment_1",
            "treatment_2"
        ],
        as_index=False
    )
    .agg(
        mean_abs_smd=(
            "abs_smd",
            "mean"
        ),
        median_abs_smd=(
            "abs_smd",
            "median"
        ),
        max_abs_smd=(
            "abs_smd",
            "max"
        ),
        pct_ge_010=(
            "abs_smd",
            lambda x:
                100 * (x >= 0.10).mean()
        ),
        pct_ge_020=(
            "abs_smd",
            lambda x:
                100 * (x >= 0.20).mean()
        )
    )
)


print("\n")
print("=" * 120)
print("BALANCE BY TREATMENT PAIR")
print("=" * 120)

print(
    pair_summary
    .round(4)
    .to_string(
        index=False
    )
)



EARLY_DPD_01_07
NUMERIC COVARIATE BALANCE
      treatment_1 treatment_2                   feature  mean_t1  mean_t2   smd  abs_smd balance_flag
friendly_reminder    pix_link            days_to_payday    14.48    14.24  0.03     0.03   GOOD <0.10
friendly_reminder    pix_link days_since_last_app_login    18.30    18.01  0.01     0.01   GOOD <0.10
friendly_reminder    pix_link   outstanding_balance_brl   851.11   846.24  0.01     0.01   GOOD <0.10
friendly_reminder    pix_link        account_age_months     9.28     9.22  0.01     0.01   GOOD <0.10
friendly_reminder    pix_link         balance_to_salary     0.33     0.33  0.01     0.01   GOOD <0.10
friendly_reminder    pix_link      n_prior_transactions     8.99     8.92  0.01     0.01   GOOD <0.10
friendly_reminder    pix_link       payday_day_of_month    12.81    12.87 -0.01     0.01   GOOD <0.10
friendly_reminder    pix_link           n_msgs_last_14d     0.83     0.83  0.00     0.00   GOOD <0.10
friendly_reminder    pix_link   custom

## Condicionando às variáveis pré-tratamento observadas, os grupos apresentam excelente balanceamento e forte suporte comum.

##__________________________________________________________________________________________________________________________________________

##__________________________________________________________________________________________________________________________________________

In [15]:
# ============================================================
# 05A — RAW TREATMENT OUTCOME ANALYSIS
# ============================================================
#
# Question:
# Among empirically well-balanced treatment groups,
# what outcomes were observed after each treatment?
#
# Outcomes:
#   1. paid_within_72h
#   2. amount_paid_brl
#
# IMPORTANT:
# These are still OBSERVATIONAL treatment comparisons.
# We are NOT yet calling them causal effects.
# ============================================================

import pandas as pd
import numpy as np
from itertools import combinations


# ============================================================
# 1. BASE
# ============================================================

outcome = support.copy()

outcome["paid_within_72h"] = (
    pd.to_numeric(
        outcome["paid_within_72h"],
        errors="coerce"
    )
)

outcome["amount_paid_brl"] = (
    pd.to_numeric(
        outcome["amount_paid_brl"],
        errors="coerce"
    )
    .fillna(0)
)


# ============================================================
# 2. BASIC OUTCOME SANITY CHECK
# ============================================================

print("=" * 110)
print("OUTCOME SANITY CHECK")
print("=" * 110)

print(f"Rows             : {len(outcome):,}")
print(
    f"Missing payment flag : "
    f"{outcome['paid_within_72h'].isna().sum():,}"
)

print(
    f"Payment events       : "
    f"{outcome['paid_within_72h'].eq(1).sum():,}"
)

print(
    f"Total amount paid    : "
    f"R$ {outcome['amount_paid_brl'].sum():,.2f}"
)

print(
    f"Positive amount rows : "
    f"{outcome['amount_paid_brl'].gt(0).sum():,}"
)


# ============================================================
# 3. CONSISTENCY BETWEEN PAYMENT FLAG AND AMOUNT
# ============================================================

print("\n" + "=" * 110)
print("PAYMENT FLAG × AMOUNT CONSISTENCY")
print("=" * 110)

consistency = pd.crosstab(
    outcome["paid_within_72h"],
    outcome["amount_paid_brl"].gt(0),
    margins=True
)

print(consistency)


# ============================================================
# 4. DEFINE REGIONS
# ============================================================

outcome_regions = {

    "EARLY_DPD_01_07": {
        "min_dpd": 1,
        "max_dpd": 7,
        "treatments": [
            "friendly_reminder",
            "pix_link"
        ]
    },

    "MID_DPD_08_30": {
        "min_dpd": 8,
        "max_dpd": 30,
        "treatments": [
            "friendly_reminder",
            "pix_link",
            "urgent_reminder"
        ]
    },

    "LATE_DPD_31_60": {
        "min_dpd": 31,
        "max_dpd": 60,
        "treatments": [
            "pix_link",
            "urgent_reminder",
            "discount_offer"
        ]
    }
}


# ============================================================
# 5. TREATMENT OUTCOME SUMMARY
# ============================================================

summary_tables = {}

for region_name, config in outcome_regions.items():

    region = outcome[
        outcome["days_past_due"].between(
            config["min_dpd"],
            config["max_dpd"]
        )
        &
        outcome["template"].isin(
            config["treatments"]
        )
    ].copy()

    summary = (
        region
        .groupby(
            "template",
            as_index=False
        )
        .agg(
            messages=(
                "message_id",
                "count"
            ),

            customers=(
                "customer_id",
                "nunique"
            ),

            payment_events=(
                "paid_within_72h",
                "sum"
            ),

            payment_rate=(
                "paid_within_72h",
                "mean"
            ),

            total_recovery_brl=(
                "amount_paid_brl",
                "sum"
            ),

            recovery_per_message_brl=(
                "amount_paid_brl",
                "mean"
            ),

            median_recovery_per_message_brl=(
                "amount_paid_brl",
                "median"
            ),

            mean_dpd=(
                "days_past_due",
                "mean"
            ),

            mean_balance_brl=(
                "outstanding_balance_brl",
                "mean"
            )
        )
    )

    summary["payment_rate_pct"] = (
        100 * summary["payment_rate"]
    )

    summary["recovery_per_customer_brl"] = (
        summary["total_recovery_brl"]
        /
        summary["customers"]
    )

    summary_tables[
        region_name
    ] = summary


    print("\n")
    print("=" * 120)
    print(region_name)
    print("RAW OUTCOME BY TREATMENT")
    print("=" * 120)

    print(
        summary[
            [
                "template",
                "messages",
                "customers",
                "payment_events",
                "payment_rate_pct",
                "total_recovery_brl",
                "recovery_per_message_brl",
                "recovery_per_customer_brl",
                "mean_dpd",
                "mean_balance_brl"
            ]
        ]
        .round(2)
        .to_string(
            index=False
        )
    )


# ============================================================
# 6. OUTCOME AMONG PAYERS ONLY
#
# Important distinction:
#
# E[recovery | message]
# versus
# E[recovery | payment occurred]
#
# ============================================================

for region_name, config in outcome_regions.items():

    region = outcome[
        outcome["days_past_due"].between(
            config["min_dpd"],
            config["max_dpd"]
        )
        &
        outcome["template"].isin(
            config["treatments"]
        )
    ].copy()

    payers = region[
        region["amount_paid_brl"] > 0
    ].copy()

    payer_summary = (
        payers
        .groupby(
            "template",
            as_index=False
        )
        .agg(
            positive_payment_events=(
                "message_id",
                "count"
            ),

            mean_amount_if_paid_brl=(
                "amount_paid_brl",
                "mean"
            ),

            median_amount_if_paid_brl=(
                "amount_paid_brl",
                "median"
            ),

            p25_amount_if_paid_brl=(
                "amount_paid_brl",
                lambda x:
                    x.quantile(.25)
            ),

            p75_amount_if_paid_brl=(
                "amount_paid_brl",
                lambda x:
                    x.quantile(.75)
            )
        )
    )

    print("\n")
    print("=" * 120)
    print(region_name)
    print("RECOVERY CONDITIONAL ON POSITIVE PAYMENT")
    print("=" * 120)

    print(
        payer_summary
        .round(2)
        .to_string(
            index=False
        )
    )


# ============================================================
# 7. PAIRWISE DIFFERENCES
#
# These are RAW observational differences.
#
# NOT causal effects yet.
# ============================================================

pairwise_rows = []

for region_name, config in outcome_regions.items():

    region = outcome[
        outcome["days_past_due"].between(
            config["min_dpd"],
            config["max_dpd"]
        )
        &
        outcome["template"].isin(
            config["treatments"]
        )
    ].copy()

    for t1, t2 in combinations(
        config["treatments"],
        2
    ):

        a = region[
            region["template"] == t1
        ]

        b = region[
            region["template"] == t2
        ]

        payment_rate_a = (
            a["paid_within_72h"].mean()
        )

        payment_rate_b = (
            b["paid_within_72h"].mean()
        )

        recovery_a = (
            a["amount_paid_brl"].mean()
        )

        recovery_b = (
            b["amount_paid_brl"].mean()
        )

        pairwise_rows.append({

            "region":
                region_name,

            "treatment_A":
                t1,

            "treatment_B":
                t2,

            "n_A":
                len(a),

            "n_B":
                len(b),

            "payment_rate_A_pct":
                100 * payment_rate_a,

            "payment_rate_B_pct":
                100 * payment_rate_b,

            "payment_rate_diff_pp_A_minus_B":
                100 * (
                    payment_rate_a
                    -
                    payment_rate_b
                ),

            "recovery_per_msg_A":
                recovery_a,

            "recovery_per_msg_B":
                recovery_b,

            "recovery_diff_A_minus_B":
                recovery_a
                -
                recovery_b
        })


pairwise_outcomes = pd.DataFrame(
    pairwise_rows
)


print("\n")
print("=" * 120)
print("RAW PAIRWISE TREATMENT DIFFERENCES")
print("=" * 120)

print(
    pairwise_outcomes
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 8. MONTH STABILITY
#
# Very important:
# We do not want an apparent treatment advantage that exists
# only because of one month.
# ============================================================

for region_name, config in outcome_regions.items():

    region = outcome[
        outcome["days_past_due"].between(
            config["min_dpd"],
            config["max_dpd"]
        )
        &
        outcome["template"].isin(
            config["treatments"]
        )
    ].copy()

    monthly = (
        region
        .groupby(
            [
                "month",
                "template"
            ],
            as_index=False
        )
        .agg(
            messages=(
                "message_id",
                "count"
            ),

            payment_rate=(
                "paid_within_72h",
                "mean"
            ),

            recovery_per_message_brl=(
                "amount_paid_brl",
                "mean"
            ),

            total_recovery_brl=(
                "amount_paid_brl",
                "sum"
            )
        )
    )

    monthly["payment_rate_pct"] = (
        100 * monthly["payment_rate"]
    )

    print("\n")
    print("=" * 120)
    print(region_name)
    print("OUTCOME BY MONTH × TREATMENT")
    print("=" * 120)

    print(
        monthly[
            [
                "month",
                "template",
                "messages",
                "payment_rate_pct",
                "recovery_per_message_brl",
                "total_recovery_brl"
            ]
        ]
        .round(2)
        .to_string(
            index=False
        )
    )


# ============================================================
# 9. DPD SUB-BUCKET STABILITY
#
# Even within our regions, check whether treatment differences
# are being driven by a narrower DPD composition.
# ============================================================

outcome["dpd_subbucket"] = pd.cut(
    outcome["days_past_due"],
    bins=[
        0,
        7,
        15,
        30,
        45,
        60
    ],
    labels=[
        "01-07",
        "08-15",
        "16-30",
        "31-45",
        "46-60"
    ]
)


dpd_stability = (
    outcome[
        outcome["days_past_due"].between(
            1,
            60
        )
    ]
    .groupby(
        [
            "dpd_subbucket",
            "template"
        ],
        observed=True,
        as_index=False
    )
    .agg(
        messages=(
            "message_id",
            "count"
        ),

        payment_rate=(
            "paid_within_72h",
            "mean"
        ),

        recovery_per_message_brl=(
            "amount_paid_brl",
            "mean"
        )
    )
)

dpd_stability["payment_rate_pct"] = (
    100 * dpd_stability["payment_rate"]
)


print("\n")
print("=" * 120)
print("OUTCOME BY DPD SUB-BUCKET × TREATMENT")
print("=" * 120)

print(
    dpd_stability[
        [
            "dpd_subbucket",
            "template",
            "messages",
            "payment_rate_pct",
            "recovery_per_message_brl"
        ]
    ]
    .round(2)
    .to_string(
        index=False
    )
)

OUTCOME SANITY CHECK
Rows             : 75,406
Missing payment flag : 0
Payment events       : 5,626
Total amount paid    : R$ 3,459,305.30
Positive amount rows : 5,626

PAYMENT FLAG × AMOUNT CONSISTENCY
amount_paid_brl  False  True    All
paid_within_72h                    
0                69780     0  69780
1                    0  5626   5626
All              69780  5626  75406


EARLY_DPD_01_07
RAW OUTCOME BY TREATMENT
         template  messages  customers  payment_events  payment_rate_pct  total_recovery_brl  recovery_per_message_brl  recovery_per_customer_brl  mean_dpd  mean_balance_brl
friendly_reminder     16583       9481            1417              8.54          927,209.73                     55.91                      97.80      3.86            851.11
         pix_link      7141       5608             733             10.26          473,887.44                     66.36                      84.50      3.85            846.24


MID_DPD_08_30
RAW OUTCOME BY TREATMENT
         t

### Pix : maior frequencia observada de pagamentos recovery per message

### 31-60 discount esta associada a mais pagadores, mas a menor valor por pagamento

# DIFERENÇAS OBSERVADAS não causalidade

Em média, quanto o resultado mudaria se a mesma população recebesse o tratamento A em vez do tratamento B? (ATE)

As diferenças permanecem depois de ajustar pelas covariavveis ?

In [19]:
# ============================================================
# 05B — DOUBLY ROBUST / AIPW
# PAIRWISE TREATMENT EFFECTS
# ============================================================
#
# Outcomes:
#   1. paid_within_72h  -> effect in probability / percentage points
#   2. amount_paid_brl  -> effect in R$ per message
#
# Method:
#   - Pairwise treatment comparisons
#   - Cross-fitting
#   - GroupKFold by customer_id
#   - Propensity model P(A | X)
#   - Outcome models E[Y | A, X]
#   - AIPW / Doubly Robust estimator
#
# IMPORTANT:
#   Effects are observational causal estimates under assumptions:
#   consistency + positivity + conditional exchangeability.
# ============================================================


import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingRegressor

from scipy.stats import norm


# ============================================================
# 1. BASE
# ============================================================

df = support.copy()

df["paid_within_72h"] = pd.to_numeric(
    df["paid_within_72h"],
    errors="coerce"
)

df["amount_paid_brl"] = pd.to_numeric(
    df["amount_paid_brl"],
    errors="coerce"
).fillna(0)


# ============================================================
# 2. FEATURES
# ============================================================

NUMERIC_FEATURES = [
    "days_past_due",
    "outstanding_balance_brl",
    "monthly_salary_brl",
    "balance_to_salary",
    "payday_day_of_month",
    "days_to_payday",
    "n_prior_transactions",
    "account_age_months",
    "days_since_last_app_login",
    "n_msgs_last_14d",
    "customer_message_number"
]

CATEGORICAL_FEATURES = [
    "state_uf",
    "dow",
    "month"
]

FEATURES = (
    NUMERIC_FEATURES
    +
    CATEGORICAL_FEATURES
)


# ============================================================
# 3. PREPROCESSOR
# ============================================================

def make_preprocessor():

    numeric_pipe = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            )
        ]
    )

    categorical_pipe = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipe,
                NUMERIC_FEATURES
            ),
            (
                "cat",
                categorical_pipe,
                CATEGORICAL_FEATURES
            )
        ]
    )


# ============================================================
# 4. PROPENSITY MODEL
# ============================================================

def make_propensity_model():

    return Pipeline(
        steps=[
            (
                "prep",
                make_preprocessor()
            ),
            (
                "model",
                HistGradientBoostingClassifier(
                    learning_rate=0.05,
                    max_iter=150,
                    max_leaf_nodes=15,
                    l2_regularization=1.0,
                    random_state=42
                )
            )
        ]
    )


# ============================================================
# 5. OUTCOME MODELS
# ============================================================

def make_payment_model():

    return Pipeline(
        steps=[
            (
                "prep",
                make_preprocessor()
            ),
            (
                "model",
                HistGradientBoostingClassifier(
                    learning_rate=0.05,
                    max_iter=150,
                    max_leaf_nodes=15,
                    l2_regularization=1.0,
                    random_state=42
                )
            )
        ]
    )


def make_recovery_model():

    return Pipeline(
        steps=[
            (
                "prep",
                make_preprocessor()
            ),
            (
                "model",
                HistGradientBoostingRegressor(
                    learning_rate=0.05,
                    max_iter=150,
                    max_leaf_nodes=15,
                    l2_regularization=1.0,
                    random_state=42
                )
            )
        ]
    )


# ============================================================
# 6. PAIRWISE AIPW
# ============================================================

def estimate_pairwise_aipw(
    data,
    treatment_A,
    treatment_B,
    outcome_col,
    outcome_type,
    n_splits=5,
    propensity_clip=0.05
):

    d = (
        data[
            data["template"].isin(
                [
                    treatment_A,
                    treatment_B
                ]
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    # ----------------------------------------
    # Binary treatment
    #
    # A = 1
    # B = 0
    #
    # Therefore:
    # estimated effect = A - B
    # ----------------------------------------

    d["T"] = (
        d["template"]
        ==
        treatment_A
    ).astype(int)

    X = d[FEATURES].copy()

    T = d["T"].to_numpy()

    Y = (
        pd.to_numeric(
            d[outcome_col],
            errors="coerce"
        )
        .fillna(0)
        .to_numpy(dtype=float)
    )

    groups = (
        d["customer_id"]
        .astype(str)
        .to_numpy()
    )


    # ----------------------------------------
    # OOF predictions
    # ----------------------------------------

    e_hat = np.full(
        len(d),
        np.nan
    )

    mu1_hat = np.full(
        len(d),
        np.nan
    )

    mu0_hat = np.full(
        len(d),
        np.nan
    )


    gkf = GroupKFold(
        n_splits=n_splits
    )


    # ----------------------------------------
    # Cross-fitting
    # ----------------------------------------

    for fold, (
        train_idx,
        test_idx
    ) in enumerate(
        gkf.split(
            X,
            T,
            groups
        ),
        start=1
    ):

        X_train = X.iloc[
            train_idx
        ]

        X_test = X.iloc[
            test_idx
        ]

        T_train = T[
            train_idx
        ]

        Y_train = Y[
            train_idx
        ]


        # ====================================
        # PROPENSITY
        # P(T = 1 | X)
        # ====================================

        propensity_model = (
            make_propensity_model()
        )

        propensity_model.fit(
            X_train,
            T_train
        )

        propensity_classes = (
            propensity_model
            .named_steps["model"]
            .classes_
        )

        class_1_position = (
            np.where(
                propensity_classes == 1
            )[0][0]
        )

        e_hat[
            test_idx
        ] = (
            propensity_model
            .predict_proba(
                X_test
            )[
                :,
                class_1_position
            ]
        )


        # ====================================
        # OUTCOME MODEL — T = 1
        # ====================================

        if outcome_type == "binary":

            model_1 = (
                make_payment_model()
            )

        elif outcome_type == "continuous":

            model_1 = (
                make_recovery_model()
            )

        else:

            raise ValueError(
                "outcome_type must be "
                "'binary' or 'continuous'"
            )


        mask_1 = (
            T_train == 1
        )

        model_1.fit(
            X_train.iloc[
                np.where(mask_1)[0]
            ],
            Y_train[
                mask_1
            ]
        )


        if outcome_type == "binary":

            mu1_hat[
                test_idx
            ] = (
                model_1
                .predict_proba(
                    X_test
                )[:, 1]
            )

        else:

            mu1_hat[
                test_idx
            ] = (
                model_1
                .predict(
                    X_test
                )
            )


        # ====================================
        # OUTCOME MODEL — T = 0
        # ====================================

        model_0 = (
            make_payment_model()
            if outcome_type == "binary"
            else make_recovery_model()
        )

        mask_0 = (
            T_train == 0
        )

        model_0.fit(
            X_train.iloc[
                np.where(mask_0)[0]
            ],
            Y_train[
                mask_0
            ]
        )


        if outcome_type == "binary":

            mu0_hat[
                test_idx
            ] = (
                model_0
                .predict_proba(
                    X_test
                )[:, 1]
            )

        else:

            mu0_hat[
                test_idx
            ] = (
                model_0
                .predict(
                    X_test
                )
            )


    # ========================================================
    # 7. CHECK OOF PREDICTIONS
    # ========================================================

    if (
        np.isnan(e_hat).any()
        or
        np.isnan(mu1_hat).any()
        or
        np.isnan(mu0_hat).any()
    ):

        raise RuntimeError(
            "Missing OOF predictions."
        )


    # ========================================================
    # 8. PROPENSITY CLIPPING
    # ========================================================

    e_raw = e_hat.copy()

    e_hat = np.clip(
        e_hat,
        propensity_clip,
        1 - propensity_clip
    )


    # ========================================================
    # 9. AIPW PSEUDO-OUTCOME
    # ========================================================

    psi = (

        # Outcome-model component
        mu1_hat
        -
        mu0_hat

        # Correction for treated
        +
        T
        *
        (
            Y
            -
            mu1_hat
        )
        /
        e_hat

        # Correction for control
        -
        (
            1 - T
        )
        *
        (
            Y
            -
            mu0_hat
        )
        /
        (
            1 - e_hat
        )
    )


    # ========================================================
    # 10. ATE
    # ========================================================

    ate = np.mean(
        psi
    )


    # ========================================================
    # 11. STANDARD ERROR
    #
    # This is row-level influence-function SE.
    # We will additionally report customer-clustered SE below.
    # ========================================================

    influence = (
        psi
        -
        ate
    )

    naive_se = (
        np.std(
            influence,
            ddof=1
        )
        /
        np.sqrt(
            len(influence)
        )
    )


    # ========================================================
    # 12. CUSTOMER-CLUSTERED SE
    #
    # Repeated messages from same customer are not independent.
    # Aggregate influence contribution within customer.
    # ========================================================

    influence_df = pd.DataFrame({
        "customer_id":
            d["customer_id"].values,

        "influence":
            influence
    })

    cluster_sum = (
        influence_df
        .groupby(
            "customer_id"
        )["influence"]
        .sum()
    )

    n = len(d)

    G = len(
        cluster_sum
    )

    cluster_variance = (
        G
        /
        (G - 1)
    ) * (
        np.sum(
            cluster_sum ** 2
        )
        /
        (
            n ** 2
        )
    )

    cluster_se = np.sqrt(
        cluster_variance
    )


    # ========================================================
    # 13. CONFIDENCE INTERVAL
    # ========================================================

    z = norm.ppf(
        0.975
    )

    ci_low = (
        ate
        -
        z
        *
        cluster_se
    )

    ci_high = (
        ate
        +
        z
        *
        cluster_se
    )


    # ========================================================
    # 14. RAW DIFFERENCE
    # ========================================================

    raw_A = (
        Y[
            T == 1
        ]
        .mean()
    )

    raw_B = (
        Y[
            T == 0
        ]
        .mean()
    )

    raw_diff = (
        raw_A
        -
        raw_B
    )


    # ========================================================
    # 15. PROPENSITY DIAGNOSTICS
    # ========================================================

    propensity_summary = {
        "propensity_min":
            np.min(e_raw),

        "propensity_p01":
            np.quantile(
                e_raw,
                0.01
            ),

        "propensity_p05":
            np.quantile(
                e_raw,
                0.05
            ),

        "propensity_median":
            np.median(
                e_raw
            ),

        "propensity_p95":
            np.quantile(
                e_raw,
                0.95
            ),

        "propensity_p99":
            np.quantile(
                e_raw,
                0.99
            ),

        "propensity_max":
            np.max(e_raw)
    }


    # ========================================================
    # 16. RESULT
    # ========================================================

    result = {

        "treatment_A":
            treatment_A,

        "treatment_B":
            treatment_B,

        "outcome":
            outcome_col,

        "rows":
            len(d),

        "customers":
            d[
                "customer_id"
            ].nunique(),

        "n_A":
            int(
                T.sum()
            ),

        "n_B":
            int(
                (1 - T).sum()
            ),

        "raw_A":
            raw_A,

        "raw_B":
            raw_B,

        "raw_difference":
            raw_diff,

        "aipw_ate":
            ate,

        "cluster_se":
            cluster_se,

        "ci_low":
            ci_low,

        "ci_high":
            ci_high,

        "naive_se":
            naive_se,

        **propensity_summary
    }

    return result


# ============================================================
# 17. REGION DEFINITIONS
# ============================================================

comparisons = {

    "EARLY_DPD_01_07": {

        "filter":
            (
                df["days_past_due"]
                .between(
                    1,
                    7
                )
            ),

        "pairs": [
            (
                "pix_link",
                "friendly_reminder"
            )
        ]
    },


    "MID_DPD_08_30": {

        "filter":
            (
                df["days_past_due"]
                .between(
                    8,
                    30
                )
            ),

        "pairs": [
            (
                "pix_link",
                "friendly_reminder"
            ),
            (
                "pix_link",
                "urgent_reminder"
            ),
            (
                "urgent_reminder",
                "friendly_reminder"
            )
        ]
    },


    "LATE_DPD_31_60": {

        # IMPORTANT:
        # discount exists only from July onward.

        "filter":
            (
                df["days_past_due"]
                .between(
                    31,
                    60
                )
                &
                df["month"].isin(
                    [
                        "2026-07",
                        "2026-08"
                    ]
                )
            ),

        "pairs": [
            (
                "discount_offer",
                "pix_link"
            ),
            (
                "discount_offer",
                "urgent_reminder"
            ),
            (
                "pix_link",
                "urgent_reminder"
            )
        ]
    }
}


# ============================================================
# 18. RUN AIPW
# ============================================================

results = []


for region_name, config in comparisons.items():

    region_df = df[
        config["filter"]
    ].copy()

    for (
        treatment_A,
        treatment_B
    ) in config["pairs"]:

        # ------------------------------------
        # PAYMENT
        # ------------------------------------

        payment_result = (
            estimate_pairwise_aipw(
                data=region_df,
                treatment_A=treatment_A,
                treatment_B=treatment_B,
                outcome_col="paid_within_72h",
                outcome_type="binary",
                n_splits=5,
                propensity_clip=0.05
            )
        )

        payment_result[
            "region"
        ] = region_name

        results.append(
            payment_result
        )


        # ------------------------------------
        # RECOVERY
        # ------------------------------------

        recovery_result = (
            estimate_pairwise_aipw(
                data=region_df,
                treatment_A=treatment_A,
                treatment_B=treatment_B,
                outcome_col="amount_paid_brl",
                outcome_type="continuous",
                n_splits=5,
                propensity_clip=0.05
            )
        )

        recovery_result[
            "region"
        ] = region_name

        results.append(
            recovery_result
        )


aipw_results = pd.DataFrame(
    results
)


# ============================================================
# 19. PAYMENT RESULTS
# ============================================================

payment_results = (
    aipw_results[
        aipw_results["outcome"]
        ==
        "paid_within_72h"
    ]
    .copy()
)


# Convert probability -> percentage points

for col in [
    "raw_A",
    "raw_B",
    "raw_difference",
    "aipw_ate",
    "cluster_se",
    "ci_low",
    "ci_high"
]:

    payment_results[
        col + "_pp"
    ] = (
        100
        *
        payment_results[
            col
        ]
    )


print("\n")
print("=" * 130)
print("AIPW — PAYMENT EFFECT")
print("Effect = Treatment A − Treatment B")
print("Units = percentage points")
print("=" * 130)

print(
    payment_results[
        [
            "region",
            "treatment_A",
            "treatment_B",
            "rows",
            "customers",

            "raw_A_pp",
            "raw_B_pp",

            "raw_difference_pp",

            "aipw_ate_pp",

            "cluster_se_pp",

            "ci_low_pp",
            "ci_high_pp"
        ]
    ]
    .round(3)
    .to_string(
        index=False
    )
)


# ============================================================
# 20. RECOVERY RESULTS
# ============================================================

recovery_results = (
    aipw_results[
        aipw_results["outcome"]
        ==
        "amount_paid_brl"
    ]
    .copy()
)


print("\n")
print("=" * 130)
print("AIPW — RECOVERY EFFECT")
print("Effect = Treatment A − Treatment B")
print("Units = R$ per message")
print("=" * 130)

print(
    recovery_results[
        [
            "region",
            "treatment_A",
            "treatment_B",
            "rows",
            "customers",

            "raw_A",
            "raw_B",

            "raw_difference",

            "aipw_ate",

            "cluster_se",

            "ci_low",
            "ci_high"
        ]
    ]
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 21. RAW VS ADJUSTED
# ============================================================

print("\n")
print("=" * 130)
print("RAW DIFFERENCE vs ADJUSTED AIPW — PAYMENT")
print("=" * 130)

print(
    payment_results[
        [
            "region",
            "treatment_A",
            "treatment_B",
            "raw_difference_pp",
            "aipw_ate_pp",
            "ci_low_pp",
            "ci_high_pp"
        ]
    ]
    .round(3)
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 130)
print("RAW DIFFERENCE vs ADJUSTED AIPW — RECOVERY")
print("=" * 130)

print(
    recovery_results[
        [
            "region",
            "treatment_A",
            "treatment_B",
            "raw_difference",
            "aipw_ate",
            "ci_low",
            "ci_high"
        ]
    ]
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 22. PROPENSITY DIAGNOSTICS
# ============================================================

print("\n")
print("=" * 130)
print("AIPW PROPENSITY DIAGNOSTICS")
print("=" * 130)

print(
    aipw_results[
        [
            "region",
            "treatment_A",
            "treatment_B",
            "outcome",

            "propensity_min",
            "propensity_p01",
            "propensity_p05",
            "propensity_median",
            "propensity_p95",
            "propensity_p99",
            "propensity_max"
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)



AIPW — PAYMENT EFFECT
Effect = Treatment A − Treatment B
Units = percentage points
         region     treatment_A       treatment_B  rows  customers  raw_A_pp  raw_B_pp  raw_difference_pp  aipw_ate_pp  cluster_se_pp  ci_low_pp  ci_high_pp
EARLY_DPD_01_07        pix_link friendly_reminder 23724      11019     10.27      8.54               1.72         1.79           0.42       0.96        2.62
  MID_DPD_08_30        pix_link friendly_reminder 19592       8276      7.99      5.98               2.00         1.98           0.36       1.27        2.69
  MID_DPD_08_30        pix_link   urgent_reminder 31424       9244      7.99      6.57               1.42         1.35           0.30       0.75        1.95
  MID_DPD_08_30 urgent_reminder friendly_reminder 27474       8983      6.57      5.98               0.58         0.58           0.32      -0.05        1.20
 LATE_DPD_31_60  discount_offer          pix_link  8743       4326      7.48      5.94               1.54         1.60           0

quanto mudaria o pagamento se mudasse o tratamentos ?

Os efeitos mudanca de tratamento ajustados são muito próximos das diferenças brutas

Os resultados são consistentes com um efeito incremental de Pix em DPD 1–30 e de Discount sobre payment rate em DPD 31–60

# TRATAMENTO MUDA COMPORTAMENTO POR SUBGRUPO ? 

In [21]:
# ============================================================
# 05C — HETEROGENEOUS TREATMENT EFFECTS
# SUBGROUP AIPW
# ============================================================
#
# Goal:
# Does the treatment effect vary meaningfully across
# observable customer characteristics?
#
# IMPORTANT:
# This is subgroup-level heterogeneity.
# We are NOT yet fitting an individual CATE model.
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. BASE
# ============================================================

hte = df.copy()


# ============================================================
# 2. BUILD PRE-TREATMENT SEGMENTS
# ============================================================

# ------------------------------------------------------------
# DPD
# ------------------------------------------------------------

hte["dpd_segment"] = pd.cut(
    hte["days_past_due"],
    bins=[
        0,
        7,
        15,
        30,
        45,
        60
    ],
    labels=[
        "01-07",
        "08-15",
        "16-30",
        "31-45",
        "46-60"
    ]
)


# ------------------------------------------------------------
# PAYDAY PROXIMITY
#
# Using days_to_payday already created previously.
# ------------------------------------------------------------

hte["payday_proximity"] = pd.cut(
    hte["days_to_payday"],
    bins=[
        -np.inf,
        2,
        5,
        10,
        np.inf
    ],
    labels=[
        "0-2d",
        "3-5d",
        "6-10d",
        "11d+"
    ]
)


# ------------------------------------------------------------
# AFFORDABILITY
# balance / salary
# ------------------------------------------------------------

hte["affordability_segment"] = pd.cut(
    hte["balance_to_salary"],
    bins=[
        -np.inf,
        0.10,
        0.25,
        0.50,
        np.inf
    ],
    labels=[
        "<=10%",
        "10-25%",
        "25-50%",
        ">50%"
    ]
)


# ------------------------------------------------------------
# BALANCE
#
# Quantiles based on the historical population.
# ------------------------------------------------------------

hte["balance_segment"] = pd.qcut(
    hte["outstanding_balance_brl"],
    q=4,
    duplicates="drop"
)


# ------------------------------------------------------------
# SALARY
# ------------------------------------------------------------

hte["salary_segment"] = pd.qcut(
    hte["monthly_salary_brl"],
    q=4,
    duplicates="drop"
)


# ------------------------------------------------------------
# LOGIN RECENCY
# ------------------------------------------------------------

hte["login_segment"] = pd.cut(
    hte["days_since_last_app_login"],
    bins=[
        -np.inf,
        7,
        15,
        30,
        np.inf
    ],
    labels=[
        "<=7d",
        "8-15d",
        "16-30d",
        "30d+"
    ]
)


# ------------------------------------------------------------
# CONTACT PRESSURE
# ------------------------------------------------------------

hte["pressure_segment"] = pd.cut(
    hte["n_msgs_last_14d"],
    bins=[
        -np.inf,
        0,
        2,
        4,
        np.inf
    ],
    labels=[
        "0",
        "1-2",
        "3-4",
        "5+"
    ]
)


# ============================================================
# 3. COMPARISONS
# ============================================================

hte_comparisons = {

    "EARLY_DPD_01_07": {
        "filter":
            hte["days_past_due"].between(
                1,
                7
            ),

        "pairs": [
            (
                "pix_link",
                "friendly_reminder"
            )
        ]
    },


    "MID_DPD_08_30": {
        "filter":
            hte["days_past_due"].between(
                8,
                30
            ),

        "pairs": [
            (
                "pix_link",
                "friendly_reminder"
            ),
            (
                "pix_link",
                "urgent_reminder"
            )
        ]
    },


    "LATE_DPD_31_60": {
        "filter":
            (
                hte["days_past_due"]
                .between(
                    31,
                    60
                )
                &
                hte["month"].isin(
                    [
                        "2026-07",
                        "2026-08"
                    ]
                )
            ),

        "pairs": [
            (
                "discount_offer",
                "pix_link"
            ),
            (
                "discount_offer",
                "urgent_reminder"
            )
        ]
    }
}


# ============================================================
# 4. SEGMENTS TO TEST
# ============================================================

SEGMENTS = [

    "dpd_segment",

    "payday_proximity",

    "affordability_segment",

    "balance_segment",

    "salary_segment",

    "login_segment",

    "pressure_segment"
]


# ============================================================
# 5. FUNCTION — SUBGROUP AIPW
# ============================================================

def run_subgroup_aipw(
    data,
    region_name,
    treatment_A,
    treatment_B,
    segment_col,
    min_rows=500,
    min_treatment_rows=100
):

    results = []

    segment_values = (
        data[segment_col]
        .dropna()
        .unique()
    )

    for segment_value in segment_values:

        subgroup = (
            data[
                data[segment_col]
                ==
                segment_value
            ]
            .copy()
        )


        # ----------------------------------------
        # Keep only relevant treatments
        # ----------------------------------------

        pair_data = subgroup[
            subgroup["template"].isin(
                [
                    treatment_A,
                    treatment_B
                ]
            )
        ].copy()


        # ----------------------------------------
        # Sample-size guardrail
        # ----------------------------------------

        if len(pair_data) < min_rows:
            continue


        treatment_counts = (
            pair_data["template"]
            .value_counts()
        )


        if (
            treatment_A
            not in treatment_counts.index
            or
            treatment_B
            not in treatment_counts.index
        ):
            continue


        if (
            treatment_counts[
                treatment_A
            ]
            <
            min_treatment_rows
            or
            treatment_counts[
                treatment_B
            ]
            <
            min_treatment_rows
        ):
            continue


        # ========================================
        # PAYMENT AIPW
        # ========================================

        payment = (
            estimate_pairwise_aipw(
                data=pair_data,
                treatment_A=treatment_A,
                treatment_B=treatment_B,
                outcome_col="paid_within_72h",
                outcome_type="binary",
                n_splits=5,
                propensity_clip=0.05
            )
        )


        # ========================================
        # RECOVERY AIPW
        # ========================================

        recovery = (
            estimate_pairwise_aipw(
                data=pair_data,
                treatment_A=treatment_A,
                treatment_B=treatment_B,
                outcome_col="amount_paid_brl",
                outcome_type="continuous",
                n_splits=5,
                propensity_clip=0.05
            )
        )


        results.append({

            "region":
                region_name,

            "comparison":
                (
                    f"{treatment_A} - "
                    f"{treatment_B}"
                ),

            "segment_variable":
                segment_col,

            "segment":
                str(segment_value),

            "rows":
                len(pair_data),

            "customers":
                pair_data[
                    "customer_id"
                ].nunique(),

            "n_A":
                treatment_counts[
                    treatment_A
                ],

            "n_B":
                treatment_counts[
                    treatment_B
                ],


            # ------------------------------------
            # PAYMENT
            # ------------------------------------

            "raw_payment_diff_pp":
                100
                *
                payment[
                    "raw_difference"
                ],

            "aipw_payment_ate_pp":
                100
                *
                payment[
                    "aipw_ate"
                ],

            "payment_ci_low_pp":
                100
                *
                payment[
                    "ci_low"
                ],

            "payment_ci_high_pp":
                100
                *
                payment[
                    "ci_high"
                ],


            # ------------------------------------
            # RECOVERY
            # ------------------------------------

            "raw_recovery_diff_brl":
                recovery[
                    "raw_difference"
                ],

            "aipw_recovery_ate_brl":
                recovery[
                    "aipw_ate"
                ],

            "recovery_ci_low_brl":
                recovery[
                    "ci_low"
                ],

            "recovery_ci_high_brl":
                recovery[
                    "ci_high"
                ]
        })


    return results


# ============================================================
# 6. RUN ALL SUBGROUP ANALYSES
# ============================================================

all_hte_results = []


for (
    region_name,
    config
) in hte_comparisons.items():

    region_df = hte[
        config["filter"]
    ].copy()


    for (
        treatment_A,
        treatment_B
    ) in config["pairs"]:


        for segment_col in SEGMENTS:

            print(
                f"Running: "
                f"{region_name} | "
                f"{treatment_A} - "
                f"{treatment_B} | "
                f"{segment_col}"
            )


            subgroup_results = (
                run_subgroup_aipw(
                    data=region_df,
                    region_name=region_name,
                    treatment_A=treatment_A,
                    treatment_B=treatment_B,
                    segment_col=segment_col,
                    min_rows=500,
                    min_treatment_rows=100
                )
            )


            all_hte_results.extend(
                subgroup_results
            )


hte_results = pd.DataFrame(
    all_hte_results
)


# ============================================================
# 7. SIGNIFICANCE FLAGS
#
# Descriptive helper only.
# Do NOT interpret as multiple-testing-adjusted significance.
# ============================================================

hte_results[
    "payment_ci_excludes_zero"
] = (
    (
        hte_results[
            "payment_ci_low_pp"
        ] > 0
    )
    |
    (
        hte_results[
            "payment_ci_high_pp"
        ] < 0
    )
)


hte_results[
    "recovery_ci_excludes_zero"
] = (
    (
        hte_results[
            "recovery_ci_low_brl"
        ] > 0
    )
    |
    (
        hte_results[
            "recovery_ci_high_brl"
        ] < 0
    )
)


# ============================================================
# 8. PRINT — PAYMENT HETEROGENEITY
# ============================================================

print("\n")
print("=" * 150)
print("05C — HETEROGENEOUS TREATMENT EFFECTS — PAYMENT")
print("AIPW effect = Treatment A − Treatment B")
print("Units = percentage points")
print("=" * 150)


print(
    hte_results[
        [
            "region",
            "comparison",
            "segment_variable",
            "segment",
            "rows",
            "customers",

            "raw_payment_diff_pp",
            "aipw_payment_ate_pp",

            "payment_ci_low_pp",
            "payment_ci_high_pp",

            "payment_ci_excludes_zero"
        ]
    ]
    .sort_values(
        [
            "region",
            "comparison",
            "segment_variable",
            "segment"
        ]
    )
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 9. PRINT — RECOVERY HETEROGENEITY
# ============================================================

print("\n")
print("=" * 150)
print("05C — HETEROGENEOUS TREATMENT EFFECTS — RECOVERY")
print("AIPW effect = Treatment A − Treatment B")
print("Units = R$ per message")
print("=" * 150)


print(
    hte_results[
        [
            "region",
            "comparison",
            "segment_variable",
            "segment",
            "rows",
            "customers",

            "raw_recovery_diff_brl",
            "aipw_recovery_ate_brl",

            "recovery_ci_low_brl",
            "recovery_ci_high_brl",

            "recovery_ci_excludes_zero"
        ]
    ]
    .sort_values(
        [
            "region",
            "comparison",
            "segment_variable",
            "segment"
        ]
    )
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 10. LOOK SPECIFICALLY FOR SIGN REVERSALS
#
# Extremely important:
#
# If A > B for one subgroup
# but A < B for another subgroup,
# personalization becomes much more interesting.
# ============================================================

sign_summary = (
    hte_results
    .groupby(
        [
            "region",
            "comparison",
            "segment_variable"
        ],
        as_index=False
    )
    .agg(

        min_payment_ate_pp=(
            "aipw_payment_ate_pp",
            "min"
        ),

        max_payment_ate_pp=(
            "aipw_payment_ate_pp",
            "max"
        ),

        min_recovery_ate_brl=(
            "aipw_recovery_ate_brl",
            "min"
        ),

        max_recovery_ate_brl=(
            "aipw_recovery_ate_brl",
            "max"
        )
    )
)


sign_summary[
    "payment_sign_reversal"
] = (
    (
        sign_summary[
            "min_payment_ate_pp"
        ] < 0
    )
    &
    (
        sign_summary[
            "max_payment_ate_pp"
        ] > 0
    )
)


sign_summary[
    "recovery_sign_reversal"
] = (
    (
        sign_summary[
            "min_recovery_ate_brl"
        ] < 0
    )
    &
    (
        sign_summary[
            "max_recovery_ate_brl"
        ] > 0
    )
)


print("\n")
print("=" * 150)
print("05C — SIGN REVERSAL SUMMARY")
print("=" * 150)

print(
    sign_summary
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 11. ONLY POTENTIALLY INTERESTING HETEROGENEITY
# ============================================================

interesting = sign_summary[
    (
        sign_summary[
            "payment_sign_reversal"
        ]
    )
    |
    (
        sign_summary[
            "recovery_sign_reversal"
        ]
    )
].copy()


print("\n")
print("=" * 150)
print("SEGMENTS WITH OBSERVED AIPW SIGN REVERSAL")
print("=" * 150)

if len(interesting) == 0:

    print(
        "No sign reversals found."
    )

else:

    print(
        interesting
        .round(2)
        .to_string(
            index=False
        )
    )

Running: EARLY_DPD_01_07 | pix_link - friendly_reminder | dpd_segment
Running: EARLY_DPD_01_07 | pix_link - friendly_reminder | payday_proximity
Running: EARLY_DPD_01_07 | pix_link - friendly_reminder | affordability_segment
Running: EARLY_DPD_01_07 | pix_link - friendly_reminder | balance_segment
Running: EARLY_DPD_01_07 | pix_link - friendly_reminder | salary_segment
Running: EARLY_DPD_01_07 | pix_link - friendly_reminder | login_segment
Running: EARLY_DPD_01_07 | pix_link - friendly_reminder | pressure_segment
Running: MID_DPD_08_30 | pix_link - friendly_reminder | dpd_segment
Running: MID_DPD_08_30 | pix_link - friendly_reminder | payday_proximity
Running: MID_DPD_08_30 | pix_link - friendly_reminder | affordability_segment
Running: MID_DPD_08_30 | pix_link - friendly_reminder | balance_segment
Running: MID_DPD_08_30 | pix_link - friendly_reminder | salary_segment
Running: MID_DPD_08_30 | pix_link - friendly_reminder | login_segment
Running: MID_DPD_08_30 | pix_link - friendly_remi

In [22]:
# ============================================================
# POLICY MODEL — CONFIGURATION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error


ACTION_COL = "template"
TARGET = "amount_paid_brl"

FEATURES = [
    # Debt state
    "days_past_due",
    "outstanding_balance_brl",

    # Affordability
    "monthly_salary_brl",

    # Payday
    "payday_day_of_month",

    # Behaviour
    "days_since_last_app_login",

    # Pressure
    "n_msgs_last_14d",

    # Relationship
    "n_prior_transactions",
    "account_age_months",

    # Geography
    "state_uf",
]